#### Simple Gen AI APP Using Langchain

In [21]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [22]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

In [32]:
# loader=WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
loader=WebBaseLoader("https://www.radius.com/en-gb/")
loader

In [33]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://www.radius.com/en-gb/', 'title': 'Radius | Fleet Mobility and Connectivity Solutions', 'description': 'Radius deliver best-in-class sustainable mobility, connectivity and technology solutions and are trusted by customers in every corner of the globe.', 'language': 'en-GB'}, page_content="Radius | Fleet Mobility and Connectivity Solutions\n\n\n\n\n\n\n\n\nLoginHomeOur solutionsOur solutionsFuel cardsFuel cardsTelematicsTelematicsTelematicsVehicle trackingAsset trackingVehicle camerasKnowledge hubLoginInsuranceInsuranceVehicle solutionsVehicle solutionsTelecomsTelecomsEV chargingEV chargingEnergyEnergyExpense managementExpense managementPartnershipsOur missionOur officesCareersLeadership teamNewsESGContact usGreat Britain and Northern Ireland - EnglishFleet and connectivity solutions that scale with your businessWe support business fleets of all sizes by helping teams stay connected with customers and colleagues to run more efficiently.Find out how 

In [34]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [35]:
documents

[Document(metadata={'source': 'https://www.radius.com/en-gb/', 'title': 'Radius | Fleet Mobility and Connectivity Solutions', 'description': 'Radius deliver best-in-class sustainable mobility, connectivity and technology solutions and are trusted by customers in every corner of the globe.', 'language': 'en-GB'}, page_content='Radius | Fleet Mobility and Connectivity Solutions'),
 Document(metadata={'source': 'https://www.radius.com/en-gb/', 'title': 'Radius | Fleet Mobility and Connectivity Solutions', 'description': 'Radius deliver best-in-class sustainable mobility, connectivity and technology solutions and are trusted by customers in every corner of the globe.', 'language': 'en-GB'}, page_content='LoginHomeOur solutionsOur solutionsFuel cardsFuel cardsTelematicsTelematicsTelematicsVehicle trackingAsset trackingVehicle camerasKnowledge hubLoginInsuranceInsuranceVehicle solutionsVehicle solutionsTelecomsTelecomsEV chargingEV chargingEnergyEnergyExpense managementExpense managementPart

In [36]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [37]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [38]:
vectorstoredb

In [39]:
## Query From a vector db
# query="LangSmith has two usage limits: total traces and extended"
query="Industry-leading software and technology"
result=vectorstoredb.similarity_search(query)
result[0].page_content

"As industry experts, we also empower businesses to look to the future with our range of EV vehicles, charge points and energy solutions.Industry-leading software and technologyWe're constantly introducing innovative ways to keep our customers moving forward. This means not only offering products so you can improve efficiency but also providing monitoring solutions so you can enjoy an end-to-end service.\nOur data-driven, application-based solutions give customers hassle-free ways to quickly see what they need from wherever they are."

In [40]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [41]:
## Retrieval Chain, Document chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7f4f76babe00>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7f4f76dcfaa0>, root_client=<openai.OpenAI object at 0x7f4f76dcfdd0>, root_async_client=<openai.AsyncOpenAI object at 0x7f4f76bab4a0>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config={'run_name': '

In [42]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

'LangSmith has two usage limits: total traces and extended traces. These are the metrics being tracked on the usage graph.'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [43]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [44]:
retriever=vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [45]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f4f76dcfce0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            | Chat

In [46]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"LangSmith has two usage limits: total traces and extended"})
response['answer']

'What solutions does the company offer for the construction and infrastructure industry, based on the provided context?'

In [47]:

response

{'input': 'LangSmith has two usage limits: total traces and extended',
 'context': [Document(id='dcedef8f-a961-48d0-9663-fcaa7b267f11', metadata={'source': 'https://www.radius.com/en-gb/', 'title': 'Radius | Fleet Mobility and Connectivity Solutions', 'description': 'Radius deliver best-in-class sustainable mobility, connectivity and technology solutions and are trusted by customers in every corner of the globe.', 'language': 'en-GB'}, page_content='LoginHomeOur solutionsOur solutionsFuel cardsFuel cardsTelematicsTelematicsTelematicsVehicle trackingAsset trackingVehicle camerasKnowledge hubLoginInsuranceInsuranceVehicle solutionsVehicle solutionsTelecomsTelecomsEV chargingEV chargingEnergyEnergyExpense managementExpense managementPartnershipsOur missionOur officesCareersLeadership teamNewsESGContact usGreat Britain and Northern Ireland - EnglishFleet and connectivity solutions that scale with your businessWe support business fleets of all sizes by helping teams stay connected with cust

In [48]:
response['context']

[Document(id='dcedef8f-a961-48d0-9663-fcaa7b267f11', metadata={'source': 'https://www.radius.com/en-gb/', 'title': 'Radius | Fleet Mobility and Connectivity Solutions', 'description': 'Radius deliver best-in-class sustainable mobility, connectivity and technology solutions and are trusted by customers in every corner of the globe.', 'language': 'en-GB'}, page_content='LoginHomeOur solutionsOur solutionsFuel cardsFuel cardsTelematicsTelematicsTelematicsVehicle trackingAsset trackingVehicle camerasKnowledge hubLoginInsuranceInsuranceVehicle solutionsVehicle solutionsTelecomsTelecomsEV chargingEV chargingEnergyEnergyExpense managementExpense managementPartnershipsOur missionOur officesCareersLeadership teamNewsESGContact usGreat Britain and Northern Ireland - EnglishFleet and connectivity solutions that scale with your businessWe support business fleets of all sizes by helping teams stay connected with customers and colleagues to run more efficiently.Find out how we can help your business